# 00 · First LLM call
One Gemini call that turns invoice text into JSON.

**Setup:** add `GEMINI_API_KEY` in Colab Secrets (key icon) with *Notebook access* on. The key is never written in this notebook.

## Step 3 · Which Flash models can my key call?
The API is the source of truth for model names, not blog posts.

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

for m in client.models.list():
    if "flash" in m.name and "generateContent" in (m.supported_actions or []):
        print(m.name, "|", m.display_name)

## Step 5 · Invoice text → JSON

In [ ]:
import json
from google import genai
from google.genai import types
from google.colab import userdata

MODEL_NAME = "gemini-3.8-flash"   # from Step 3: listed by client.models.list()

INVOICE_TEXT = "ABC Traders, Invoice #INV-101, 12/11/2026, Total ₹4,500"

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

prompt = f"""Extract these fields from the invoice below.
Return JSON only, with exactly these keys: vendor, invoice_number, date, currency, total.
Use null for anything missing. No extra keys, no extra text.

Invoice:
{INVOICE_TEXT}
"""

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=prompt,
    config=types.GenerateContentConfig(response_mime_type="application/json"),
)

data = json.loads(response.text)
print(json.dumps(data, indent=2, ensure_ascii=False))

### Result (run on 18 Nov 2026)
```json
{"vendor": "ABC Traders", "invoice_number": "INV-101", "date": "12/11/2026", "currency": "₹", "total": 4500}
```
**Issues seen:** currency came back as a symbol, not ISO `INR`; `12/11/2026` is ambiguous (12 Nov or 11 Dec) and was copied without a flag. Gemini 3.5 Flash returned total as the string `"4,500"`. Fix later: a `response_schema` with ISO currency, ISO date, numeric total, and null + reason when unsure.

## Step 5.5 · Break it on purpose
Expected: `SecretNotFoundError: Secret WRONG_KEY does not exist.`

In [ ]:
from google.colab import userdata
userdata.get("WRONG_KEY")